# EPIC Clarity Care Site Hydration

This notebook hydrates the OMOP CARE_SITE table from EPIC Clarity source data.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_DEP` - Department/care site master
- `_exponent._bronze_epic_clarity_*.dbo_CLARITY_LOC` - Location/facility reference

## OMOP Fields Populated
- care_site_source_value
- care_site_name
- place_of_service_source_value
- care_site_id (surrogate key from mapping table)

In [ ]:
source = 'epic_clarity'

In [ ]:
-- Silver Layer: Transform EPIC care site data%sqlCREATE OR REPLACE TEMP VIEW care_site_silver ASSELECT    CONCAT_WS(CHR(31), 'epic_clarity', 'CLARITY_DEP', 'DEPARTMENT_ID', d.DEPARTMENT_ID) AS care_site_source_value,    d.DEPARTMENT_NAME AS care_site_name,    COALESCE(l.LOC_NAME, '') AS place_of_service_source_value,    CURRENT_TIMESTAMP() AS updated_tspFROM _exponent._bronze_epic_clarity.clarity_dep dLEFT JOIN _exponent._bronze_epic_clarity.clarity_loc l    ON d.LOC_ID = l.LOC_IDWHERE d.DEPARTMENT_ID IS NOT NULL

In [ ]:
-- Merge into Silver Layer
%sql
MERGE INTO _exponent.omop_silver.care_site AS target
USING care_site_silver AS source
ON target.care_site_source_value = source.care_site_source_value

WHEN MATCHED AND NOT (
    target.care_site_name <=> source.care_site_name
    AND target.place_of_service_source_value <=> source.place_of_service_source_value
)
THEN UPDATE SET
    target.care_site_name = source.care_site_name,
    target.place_of_service_source_value = source.place_of_service_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    care_site_source_value,
    care_site_name,
    place_of_service_source_value,
    updated_tsp
)
VALUES (
    source.care_site_source_value,
    source.care_site_name,
    source.place_of_service_source_value,
    source.updated_tsp
)

In [ ]:
-- Populate mapping table
%sql
INSERT INTO _exponent.omop_mapping.source_to_care_site (
    source_system,
    care_site_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    'epic_clarity' AS source_system,
    s.care_site_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    s.updated_tsp AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT care_site_source_value, updated_tsp
    FROM _exponent.omop_silver.care_site
    WHERE care_site_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_care_site x
    ON s.care_site_source_value = x.care_site_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
-- Gold Layer: Join with mapping to get care_site_id
%sql
CREATE OR REPLACE TEMP VIEW care_site_gold AS
SELECT
    m.care_site_id,
    s.care_site_name,
    s.place_of_service_source_value,
    s.updated_tsp
FROM _exponent.omop_silver.care_site s
INNER JOIN _exponent.omop_mapping.source_to_care_site m
    ON s.care_site_source_value = m.care_site_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE

In [ ]:
-- Merge into Gold Layer (OMOP)
%sql
MERGE INTO _exponent.omop.care_site AS target
USING care_site_gold AS source
ON target.care_site_id = source.care_site_id

WHEN MATCHED AND NOT (
    target.care_site_name <=> source.care_site_name
    AND target.place_of_service_source_value <=> source.place_of_service_source_value
)
THEN UPDATE SET
    target.care_site_name = source.care_site_name,
    target.place_of_service_source_value = source.place_of_service_source_value

WHEN NOT MATCHED THEN INSERT (
    care_site_id,
    care_site_name,
    place_of_service_source_value
)
VALUES (
    source.care_site_id,
    source.care_site_name,
    source.place_of_service_source_value
)